# GRPO reward-scaling experiment: Colab A100 runner

One-click pipeline for the AISTATS LLM experiment: Qwen2.5-0.5B-Instruct + LoRA,
TRL `GRPOTrainer`, comparing GRPO / Dr.GRPO / global normalization reward scaling
under a per-family reward-scale asymmetry (`k`).

**Run cells top to bottom.** The last cell (full 7-config sweep) is optional and
gated -- only run it after reviewing the pilot's PASS/FAIL verdict.

All results are saved to Google Drive (`grpo_llm_results/`) after each run, so the
sweep can resume from a Colab disconnect without losing completed work.

## 1. Check for a GPU

Sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` before torch ever loads --
it reduces allocator fragmentation under GRPO's bursty generate-then-backward memory
pattern, and only takes effect if set prior to CUDA context init.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess

try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False

if not has_gpu:
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type, select a GPU '
        '(A100 recommended), then Runtime > Restart session and run all cells again.'
    )

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2. Config -- edit if your fork/repo name differs

In [ ]:
GITHUB_USER = "nilay47"
GITHUB_REPO = "agenthon"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/grpo_llm_results"
REPO_DIR = f"/content/{GITHUB_REPO}"

## 3. Mount Google Drive

Also points the HuggingFace cache at Drive, so the ~1GB model download survives a
disconnect instead of being wiped along with the rest of the Colab VM's disk.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to {DRIVE_RESULTS_DIR}")

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
hf_home = os.environ['HF_HOME']
print(f"HF cache: {hf_home}")

## 4. Clone the repo

If the repo is **private**, add a `GITHUB_TOKEN` Colab secret first (key icon in the
left sidebar) with a personal access token that has `repo` scope, and grant this
notebook access to it when prompted. If the repo is public, this works with no token.

In [ ]:
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone (pulling latest instead)")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/llm

## 5. Install dependencies (Colab-safe: never touches torch/CUDA)

Installs `trl`/`peft`/`transformers`/`accelerate`/`datasets` with `--no-deps`, so pip
never resolves THEIR torch dependency against Colab's preinstalled build -- that
resolution is what previously pulled in a mismatched torch/CUDA wheel and broke
`trl.trainer.grpo_trainer` with `libcudart.so.13: cannot open shared object file`.
The lightweight, pure-Python packages those five need are installed separately with
normal resolution (they can't touch torch/CUDA either way). vLLM is intentionally
**not** installed -- `run.py` already falls back to plain HF generation automatically.

Colab also preinstalls an old `torchao` that `peft` refuses to import against
(`Found an incompatible version of torchao`) -- we never use it, so it's removed.

In [ ]:
!pip install -q --no-deps -r requirements-colab.txt
!pip install -q huggingface_hub tokenizers safetensors regex pyyaml filelock requests \
    tqdm packaging psutil pandas pyarrow dill multiprocess fsspec xxhash scipy
!pip uninstall -y -q torchao

import importlib.metadata as _metadata

for _pkg in ['torch', 'transformers', 'trl', 'peft', 'accelerate', 'datasets']:
    try:
        _ver = _metadata.version(_pkg)
    except _metadata.PackageNotFoundError:
        _ver = 'NOT INSTALLED'
    print(f"{_pkg} == {_ver}")

import torch

print(f"torch.__version__ = {torch.__version__}")
print(f"torch.version.cuda = {torch.version.cuda}")
print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")

try:
    import trl.trainer.grpo_trainer  # noqa: F401
    from peft import LoraConfig, get_peft_model  # noqa: F401
    print("trl.trainer.grpo_trainer and peft imported OK")
except Exception as e:
    raise RuntimeError(
        f"Import check failed against torch {torch.__version__} (CUDA {torch.version.cuda}). "
        f"This usually means a pip install pulled in a mismatched torch/CUDA build, or an "
        f"incompatible optional dependency (like torchao) is still installed. "
        f"Original error: {e!r}"
    ) from e

## 5b. PREFLIGHT: exercise every training config on 2 tiny steps

Loads the real model + LoRA and runs 2 GRPO steps (2 prompts x G=4, short
completions) for **each** of the four training configs used later -- `scale_rewards`
`"group"`, `"batch"`, `"none"`, and the sigma-sampling method -- including one
eval-callback call and one save to the Drive `preflight/` subfolder each. This is the
exact same code path `run.py` uses for the real runs, just at a scale that finishes in
under a minute, so a config-level break (like the torchao import failure above) surfaces
here instead of after committing to a full 150-step run.

In [ ]:
import os
import sys
sys.path.insert(0, '.')
from run import run_one

PREFLIGHT_DIR = os.path.join(DRIVE_RESULTS_DIR, 'preflight')

for _method in ['grpo', 'global', 'drgrpo', 'sigma_sampling']:
    print(f'--- preflight: {_method} ---')
    try:
        run_one(_method, k=1, seed=0, max_steps=2, out_dir=PREFLIGHT_DIR,
                prompts_per_step=2, g=4, max_completion_length=32, eval_every=1)
    except Exception as e:
        raise RuntimeError(
            f"PREFLIGHT FAILED for config '{_method}'. Original error: {e!r}"
        ) from e

print("PREFLIGHT PASSED")

## 5c. PREFLIGHT: peak GPU memory at the REAL batch shape

The config-smoke preflight above uses a tiny 2-prompt batch, which does not stress
memory. This one runs the **actual** training shape (16 prompts x G=8 = 128
completions per optimizer step) for one config, with completions forced out to the
full `max_completion_length` via `min_new_tokens` (the true worst case -- real
completions are usually shorter, so this is deliberately pessimistic), and reports
peak GPU memory. Backward is still micro-batched internally (see `run.py` /
`config.micro_batch_and_accum`), so this also doubles as a direct check that the OOM
fix actually holds at the shape that triggered it. Fails if peak reserved memory
exceeds 34 GB (A100 is 40GB; leaves headroom for the eval callback's own generation).

In [ ]:
import torch
import config

torch.cuda.reset_peak_memory_stats()
run_one('grpo', k=1, seed=0, max_steps=1, out_dir=PREFLIGHT_DIR,
        prompts_per_step=16, g=8, max_completion_length=200, eval_every=1,
        extra_grpo_kwargs=dict(generation_kwargs=dict(min_new_tokens=200)))

peak_allocated_gb = torch.cuda.max_memory_allocated() / 1024**3
peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f"peak memory allocated: {peak_allocated_gb:.2f} GB")
print(f"peak memory reserved:  {peak_reserved_gb:.2f} GB")

if peak_reserved_gb > 34:
    current_micro = config.MICRO_BATCH_COMPLETIONS
    raise RuntimeError(
        f"Peak GPU memory ({peak_reserved_gb:.2f} GB reserved) exceeds the 34 GB safety "
        f"threshold for a 40GB A100. Reduce config.MICRO_BATCH_COMPLETIONS (currently "
        f"{current_micro}) and rerun this cell."
    )
print("MEMORY PREFLIGHT PASSED")

## 6. Baseline sanity check

Runs the **untrained** model on both families' held-out sets. If a family WARNs
(reward out of [0.2, 0.7], or group std too low to give GRPO any learning signal),
read the suggested fix, edit `config.py`'s `TASK_VARIANT_A`/`TASK_VARIANT_B`, and
re-run this cell before continuing.

In [ ]:
!python baseline_eval.py --out_dir "{DRIVE_RESULTS_DIR}"

import json, os
with open(os.path.join(DRIVE_RESULTS_DIR, "baseline_eval.json")) as f:
    baseline_result = json.load(f)

## 7. Pilot: Dr.GRPO, k=1 and k=10

Runs the two pilot configs **one at a time**, saving each result to Drive
immediately after it finishes (not just at the end), so a disconnect between the
two runs doesn't lose the first one.

In [ ]:
import sys
sys.path.insert(0, '.')
from run import run_one
from pilot import PILOT_SEED, check_pass

print("=== PILOT: Dr.GRPO k=1 ===")
pilot_k1 = run_one("drgrpo", k=1, seed=PILOT_SEED, out_dir=DRIVE_RESULTS_DIR)
print(f"saved to {DRIVE_RESULTS_DIR}/drgrpo_k1_seed{PILOT_SEED}.json")

In [ ]:
print("=== PILOT: Dr.GRPO k=10 ===")
pilot_k10 = run_one("drgrpo", k=10, seed=PILOT_SEED, out_dir=DRIVE_RESULTS_DIR)
print(f"saved to {DRIVE_RESULTS_DIR}/drgrpo_k10_seed{PILOT_SEED}.json")

pilot_passed, pilot_detail = check_pass(pilot_k1, pilot_k10)
import json, os
with open(os.path.join(DRIVE_RESULTS_DIR, "pilot_summary.json"), "w") as f:
    json.dump(dict(passed=pilot_passed, detail=pilot_detail,
                   k1_eval_history=pilot_k1["eval_history"],
                   k10_eval_history=pilot_k10["eval_history"],
                   k1_wall_clock_s=pilot_k1["wall_clock_s"],
                   k10_wall_clock_s=pilot_k10["wall_clock_s"]), f, indent=2)

## 8. Final summary

In [ ]:
print('=== Baseline ===')
for fam, v in baseline_result['verdicts'].items():
    print(f"  family {fam}: {'PASS' if v['passed'] else v['detail']}")

print('\n=== Pilot: Dr.GRPO k=1 vs k=10 ===')
for label, res in [('k=1', pilot_k1), ('k=10', pilot_k10)]:
    print(f"  {label} (wall_clock={res['wall_clock_s']:.1f}s):")
    for e in res['eval_history']:
        print(f"    step={e['step']:>4}  A={e['A']:.4f}  B={e['B']:.4f}")

print(f'\n{pilot_detail}')
print('PILOT', 'PASSED' if pilot_passed else 'FAILED')

---
## 9. OPTIONAL -- full 7-config sweep

**Only run this after reviewing the pilot result above.** This trains all 7
configs (GRPO / Dr.GRPO / global-norm x k in {1,10}, plus the sigma-sampling
method at k=10) x 2 seeds -- substantially more GPU time than the pilot.

Resumable: any `(method, k, seed)` whose result file already exists in
`DRIVE_RESULTS_DIR` is skipped, so if Colab disconnects mid-sweep, just re-run
this cell (after re-running cells 1-5 to reconnect) and it picks up where it left off.

In [ ]:
from full import run_sweep, build_summary
import json, os

SEEDS = [0, 1]  # add 2 for a 3rd seed if time allows
full_results = run_sweep(SEEDS, DRIVE_RESULTS_DIR, skip_existing=True)
full_summary = build_summary(full_results, SEEDS)

for name, s in full_summary.items():
    print(f"{name}: progress_A={s['progress_A']['mean']:+.4f} "
          f"progress_B={s['progress_B']['mean']:+.4f} "
          f"B/A_ratio={s['B_over_A_ratio']['mean']:.2f}")

with open(os.path.join(DRIVE_RESULTS_DIR, "full_summary.json"), "w") as f:
    json.dump(full_summary, f, indent=2)
print(f"\nwrote {DRIVE_RESULTS_DIR}/full_summary.json")